# Notebook 05b — ModernBERT Fine-tuning (Google Colab)

Fine-tunes ModernBERT-base (2024) on the labelled email dataset using a T4 GPU on Google Colab for three-class phishing classification.

## What This Notebook Does
- Mounts Google Drive for data access and model saving
- Loads and tokenises the training dataset
- Fine-tunes ModernBERT-base for 4 epochs
- Monitors validation Macro F1 after each epoch
- Saves the best checkpoint to Google Drive
- Generates confidence scores for all training emails
- Generates confidence scores for the external BEC test set

## Prerequisites
- Google Colab account with GPU runtime (T4 recommended)
- dataset_final.csv uploaded to Google Drive at:
  phishing_detection/data/processed/
- Enable GPU: Runtime → Change runtime type → T4 GPU

## Inputs
- Google Drive: phishing_detection/data/processed/dataset_final.csv
- Google Drive: phishing_detection/data/processed/dataset_diverse.csv

## Outputs
- Google Drive: phishing_detection/models/modernbert_finetuned/
- Google Drive: phishing_detection/models/modernbert_diverse/
- Google Drive: phishing_detection/data/processed/modernbert_features.csv
- Google Drive: phishing_detection/data/processed/modernbert_diverse_features.csv
- Google Drive: phishing_detection/data/processed/bec_modernbert.csv
- Google Drive: phishing_detection/data/processed/bec_diverse_modernbert.csv

## Training Results (Diverse Dataset)
| Epoch | Train Loss | Val Loss | Macro F1 | Accuracy |
|-------|-----------|----------|----------|----------|
| 1     | 0.1327    | 0.0593   | 0.9807   | 98.1%    |
| 2     | 0.0434    | 0.0636   | 0.9867   | 98.7%    |
| 3     | 0.0153    | 0.0618   | 0.9884   | 98.9%    |
| 4     | 0.0001    | 0.0660   | 0.9878   | 98.8%    |

Best checkpoint: Epoch 3 (Macro F1 = 0.9884)

## Runtime
Approximately 12-15 minutes per training run on T4 GPU

## Note
This notebook runs on Google Colab, not locally.
Output CSV files have been downloaded and saved locally to data/processed/ for use in subsequent notebooks.
torchvision must be uninstalled before running:
pip uninstall torchvision torchaudio -y

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets accelerate -q

import pandas as pd
import numpy as np
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import Dataset

BASE_DIR = Path("/content/drive/MyDrive/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Load data
dataset = pd.read_csv(DATA_PROCESSED / "dataset_final.csv")
print(f"Dataset: {len(dataset)} emails")
print(dataset['label'].value_counts().sort_index())

# Split — same random state as all other notebooks
train_df, temp_df = train_test_split(
    dataset, test_size=0.30, random_state=42, stratify=dataset['label'])
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])

print(f"\nTrain: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Tokenize with ModernBERT
MODEL_NAME = "answerdotai/ModernBERT-base"
print(f"\nLoading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded")

def preprocess(examples):
    tokens = tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=256
    )
    tokens['labels'] = examples['label']
    return tokens

train_hf = Dataset.from_dict({
    'text': train_df['text'].tolist(),
    'label': train_df['label'].astype(int).tolist()
})
val_hf = Dataset.from_dict({
    'text': val_df['text'].tolist(),
    'label': val_df['label'].astype(int).tolist()
})
test_hf = Dataset.from_dict({
    'text': test_df['text'].tolist(),
    'label': test_df['label'].astype(int).tolist()
})

print("\nTokenizing...")
train_hf = train_hf.map(preprocess, batched=True, remove_columns=['text', 'label'])
val_hf = val_hf.map(preprocess, batched=True, remove_columns=['text', 'label'])
test_hf = test_hf.map(preprocess, batched=True, remove_columns=['text', 'label'])

train_hf.set_format('torch')
val_hf.set_format('torch')
test_hf.set_format('torch')

# Verify with forward pass test
print(f"\nColumns: {train_hf.column_names}")
sample = train_hf[0]
print(f"labels dtype: {sample['labels'].dtype}")
print(f"labels value: {sample['labels']}")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True)
model = model.to('cuda')

with torch.no_grad():
    out = model(
        input_ids=sample['input_ids'].unsqueeze(0).to('cuda'),
        attention_mask=sample['attention_mask'].unsqueeze(0).to('cuda'),
        labels=sample['labels'].unsqueeze(0).to('cuda')
    )
print(f"\nForward pass test:")
print(f"  Loss: {out.loss:.4f}")
print(f"  Logits: {out.logits}")
print("\nReady to train!" if not torch.isnan(out.loss) else "WARNING: NaN loss detected")

In [ ]:
!pip uninstall torchvision torchaudio -y -q
!pip install transformers datasets accelerate -q

# Then restart runtime and rerun Cell 1

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, predictions, average='macro')
    accuracy = (predictions == labels).mean()
    return {'macro_f1': macro_f1, 'accuracy': accuracy}

training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / "modernbert_checkpoints"),
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=100,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    compute_metrics=compute_metrics,
)

print("Starting ModernBERT fine-tuning")
trainer.train()

print("\nFine-tuning complete!")
metrics = trainer.evaluate()
print("\nFinal validation metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
save_path = str(MODELS_DIR / "modernbert_finetuned")
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to: {save_path}")

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

# Load saved model
save_path = str(MODELS_DIR / "modernbert_finetuned")
print("Loading fine-tuned ModernBERT...")
model_loaded = AutoModelForSequenceClassification.from_pretrained(save_path)
tokenizer_loaded = AutoTokenizer.from_pretrained(save_path)
model_loaded = model_loaded.to('cuda')
model_loaded.eval()
print("Model loaded")

# Load full dataset
dataset = pd.read_csv(DATA_PROCESSED / "dataset_final.csv")
print(f"Processing {len(dataset)} emails...")

# Process in batches
results = []
batch_size = 32

for i in tqdm(range(0, len(dataset), batch_size), desc="Generating scores"):
    batch_texts = dataset['text'].iloc[i:i+batch_size].tolist()
    batch_labels = dataset['label'].iloc[i:i+batch_size].tolist()

    # Tokenize
    inputs = tokenizer_loaded(
        batch_texts,
        truncation=True,
        padding='max_length',
        max_length=256,
        return_tensors='pt'
    ).to('cuda')

    # Get confidence scores
    with torch.no_grad():
        outputs = model_loaded(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()

    for j, (prob, label) in enumerate(zip(probs, batch_labels)):
        results.append({
            'email_idx': i + j,
            'true_label': label,
            'llm_label': int(np.argmax(prob)),
            'conf_legitimate': float(prob[0]),
            'conf_human_phishing': float(prob[1]),
            'conf_ai_phishing': float(prob[2])
        })

# Save results
results_df = pd.DataFrame(results)
save_path_csv = DATA_PROCESSED / "modernbert_features.csv"
results_df.to_csv(save_path_csv, index=False)

print(f"\nDone! {len(results_df)} emails processed")
print(f"\nPrediction distribution:")
print(results_df['llm_label'].value_counts().sort_index())
print(f"\nSaved to: {save_path_csv}")

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

BASE_DIR = Path("/content/drive/MyDrive/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"

# Load fine-tuned ModernBERT
model_path = str(MODELS_DIR / "modernbert_finetuned")
model = AutoModelForSequenceClassification.from_pretrained(model_path).to('cuda')
tokenizer = AutoTokenizer.from_pretrained(model_path)
model.eval()
print("ModernBERT loaded")

# Load BEC dataset
bec = pd.read_csv(DATA_PROCESSED / "bec_dataset.csv")
bec['text'] = bec['subject'].astype(str) + " " + bec['body'].astype(str)
bec = bec.dropna(subset=['label'])
print(f"BEC emails: {len(bec)}")

# Generate scores
results = []
batch_size = 32
texts = bec['text'].tolist()

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    inputs = tokenizer(batch, truncation=True, padding='max_length',
                       max_length=256, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    for prob in probs:
        results.append({
            'llm_label': int(np.argmax(prob)),
            'conf_legitimate': float(prob[0]),
            'conf_human_phishing': float(prob[1]),
            'conf_ai_phishing': float(prob[2])
        })

bec_scores = pd.DataFrame(results)
bec_scores.to_csv(DATA_PROCESSED / "bec_modernbert.csv", index=False)

print(f"\nDone! {len(bec_scores)} emails scored")
print(f"\nModernBERT predictions on BEC (unseen Gemini phishing):")
print(bec_scores['llm_label'].value_counts().sort_index())
print("\nLabel meaning: 0=legitimate, 1=human phishing, 2=AI phishing")

In [ ]:
from pathlib import Path
model_path = Path("/content/drive/MyDrive/phishing_detection/models/modernbert_finetuned")
print(f"Folder exists: {model_path.exists()}")
if model_path.exists():
    print("Files in model folder:")
    for f in model_path.iterdir():
        print(f"  {f.name}")

In [ ]:
from pathlib import Path

models_dir = Path("/content/drive/MyDrive/phishing_detection/models")
print(f"Models folder exists: {models_dir.exists()}\n")

if models_dir.exists():
    print("Contents of models folder:")
    for f in models_dir.iterdir():
        print(f"  {f.name}")
else:
    print("Searching entire Drive for ModernBERT files...")
    drive_root = Path("/content/drive/MyDrive/phishing_detection")
    for f in drive_root.rglob("*modernbert*"):
        print(f"  Found: {f}")
    for f in drive_root.rglob("*.safetensors"):
        print(f"  Found: {f}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import Dataset

BASE_DIR = Path("/content/drive/MyDrive/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)  # Create models folder explicitly

print(f"Models folder created: {MODELS_DIR.exists()}")

# Load and split data
dataset = pd.read_csv(DATA_PROCESSED / "dataset_final.csv")
train_df, temp_df = train_test_split(
    dataset, test_size=0.30, random_state=42, stratify=dataset['label'])
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Tokenize
MODEL_NAME = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    tokens = tokenizer(examples['text'], truncation=True,
                       padding='max_length', max_length=256)
    tokens['labels'] = examples['label']
    return tokens

train_hf = Dataset.from_dict({'text': train_df['text'].tolist(),
                              'label': train_df['label'].astype(int).tolist()})
val_hf = Dataset.from_dict({'text': val_df['text'].tolist(),
                            'label': val_df['label'].astype(int).tolist()})

train_hf = train_hf.map(preprocess, batched=True, remove_columns=['text','label'])
val_hf = val_hf.map(preprocess, batched=True, remove_columns=['text','label'])
train_hf.set_format('torch')
val_hf.set_format('torch')

# Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'macro_f1': f1_score(labels, preds, average='macro'),
            'accuracy': (preds == labels).mean()}

training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / "modernbert_checkpoints"),
    num_train_epochs=4, per_device_train_batch_size=16,
    per_device_eval_batch_size=32, warmup_steps=200,
    weight_decay=0.01, learning_rate=2e-5,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    logging_steps=100, fp16=True, report_to="none")

trainer = Trainer(model=model, args=training_args,
                  train_dataset=train_hf, eval_dataset=val_hf,
                  compute_metrics=compute_metrics)

print("Training ModernBERT")
trainer.train()

# Save and verify
save_path = str(MODELS_DIR / "modernbert_finetuned")
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

# Verify save worked
saved = Path(save_path)
print(f"\nSave verification:")
print(f"  Folder exists: {saved.exists()}")
if saved.exists():
    for f in saved.iterdir():
        print(f"  {f.name}")

In [ ]:
!pip uninstall torchvision torchaudio -y -q
print("torchvision removed - now restart runtime")

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

BASE_DIR = Path("/content/drive/MyDrive/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"

# Load fine-tuned ModernBERT
model_path = str(MODELS_DIR / "modernbert_finetuned")
model = AutoModelForSequenceClassification.from_pretrained(
    model_path, local_files_only=True).to('cuda')
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model.eval()
print("ModernBERT loaded")

# Load BEC dataset
bec = pd.read_csv(DATA_PROCESSED / "bec_dataset.csv")
bec['text'] = bec['subject'].astype(str) + " " + bec['body'].astype(str)
bec = bec.dropna(subset=['label'])
print(f"BEC emails: {len(bec)}")

# Generate scores
results = []
batch_size = 32
texts = bec['text'].tolist()

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    inputs = tokenizer(batch, truncation=True, padding='max_length',
                       max_length=256, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    for prob in probs:
        results.append({
            'llm_label': int(np.argmax(prob)),
            'conf_legitimate': float(prob[0]),
            'conf_human_phishing': float(prob[1]),
            'conf_ai_phishing': float(prob[2])
        })

bec_scores = pd.DataFrame(results)
bec_scores.to_csv(DATA_PROCESSED / "bec_modernbert.csv", index=False)

print(f"\nDone! {len(bec_scores)} emails scored")
print(f"\nModernBERT predictions on unseen Gemini BEC phishing:")
print(bec_scores['llm_label'].value_counts().sort_index())
print("\n0=legitimate, 1=human phishing, 2=AI phishing")

# Key metric: how many flagged as phishing (1 or 2) vs missed (0)
flagged = (bec_scores['llm_label'] != 0).sum()
print(f"\nFlagged as phishing: {flagged}/{len(bec_scores)} ({flagged/len(bec_scores)*100:.1f}%)")
print(f"Missed (called legitimate): {(bec_scores['llm_label']==0).sum()}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip uninstall torchvision torchaudio -y -q
!pip install transformers datasets accelerate -q

import pandas as pd
import numpy as np
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import Dataset

BASE_DIR = Path("/content/drive/MyDrive/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")

# Load CLEANED dataset
dataset = pd.read_csv(DATA_PROCESSED / "dataset_cleaned.csv")
print(f"Cleaned dataset: {len(dataset)} emails")
print(dataset['label'].value_counts().sort_index())

# Split
train_df, temp_df = train_test_split(
    dataset, test_size=0.30, random_state=42, stratify=dataset['label'])
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])

print(f"\nTrain: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Tokenize
MODEL_NAME = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    tokens = tokenizer(examples['text'], truncation=True,
                       padding='max_length', max_length=256)
    tokens['labels'] = examples['label']
    return tokens

train_hf = Dataset.from_dict({'text': train_df['text'].tolist(),
                               'label': train_df['label'].astype(int).tolist()})
val_hf = Dataset.from_dict({'text': val_df['text'].tolist(),
                             'label': val_df['label'].astype(int).tolist()})

train_hf = train_hf.map(preprocess, batched=True, remove_columns=['text','label'])
val_hf = val_hf.map(preprocess, batched=True, remove_columns=['text','label'])
train_hf.set_format('torch')
val_hf.set_format('torch')

# Verify
sample = train_hf[0]
print(f"\nDataset check:")
print(f"  Columns: {train_hf.column_names}")
print(f"  labels dtype: {sample['labels'].dtype}")

# Forward pass test
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True).to('cuda')

with torch.no_grad():
    out = model(
        input_ids=sample['input_ids'].unsqueeze(0).to('cuda'),
        attention_mask=sample['attention_mask'].unsqueeze(0).to('cuda'),
        labels=sample['labels'].unsqueeze(0).to('cuda'))
print(f"\nForward pass loss: {out.loss:.4f}")
print("Ready to train!" if not torch.isnan(out.loss) else "WARNING: NaN")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, predictions, average='macro')
    accuracy = (predictions == labels).mean()
    return {'macro_f1': macro_f1, 'accuracy': accuracy}

training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / "modernbert_cleaned_checkpoints"),
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=100,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    compute_metrics=compute_metrics,
)

print("Training ModernBERT on cleaned dataset")
trainer.train()

print("\nTraining complete!")
metrics = trainer.evaluate()
print("\nFinal validation metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

# Save
save_path = str(MODELS_DIR / "modernbert_cleaned")
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

# Verify save
from pathlib import Path
saved = Path(save_path)
print(f"\nSave verification:")
print(f"  Folder exists: {saved.exists()}")
for f in saved.iterdir():
    print(f"  {f.name}")

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

BASE_DIR = Path("/content/drive/MyDrive/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"

# Load cleaned model
model_path = str(MODELS_DIR / "modernbert_cleaned")
model_inf = AutoModelForSequenceClassification.from_pretrained(
    model_path, local_files_only=True).to('cuda')
tokenizer_inf = AutoTokenizer.from_pretrained(
    model_path, local_files_only=True)
model_inf.eval()
print("Cleaned ModernBERT loaded")

# Load cleaned dataset
dataset = pd.read_csv(DATA_PROCESSED / "dataset_cleaned.csv")
print(f"Processing {len(dataset)} emails")

results = []
batch_size = 32

for i in tqdm(range(0, len(dataset), batch_size), desc="Scoring"):
    batch_texts = dataset['text'].iloc[i:i+batch_size].tolist()
    batch_labels = dataset['label'].iloc[i:i+batch_size].tolist()
    inputs = tokenizer_inf(batch_texts, truncation=True,
                           padding='max_length', max_length=256,
                           return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model_inf(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    for j, (prob, label) in enumerate(zip(probs, batch_labels)):
        results.append({
            'email_idx': i+j, 'true_label': label,
            'llm_label': int(np.argmax(prob)),
            'conf_legitimate': float(prob[0]),
            'conf_human_phishing': float(prob[1]),
            'conf_ai_phishing': float(prob[2])
        })

results_df = pd.DataFrame(results)
results_df.to_csv(DATA_PROCESSED / "modernbert_cleaned_features.csv", index=False)

print(f"\nDone! {len(results_df)} emails scored")
print(f"\nPrediction distribution:")
print(results_df['llm_label'].value_counts().sort_index())

# Also score BEC external test
bec = pd.read_csv(DATA_PROCESSED / "bec_dataset.csv")
bec['text'] = bec['subject'].astype(str) + " " + bec['body'].astype(str)
bec = bec.dropna(subset=['label']).reset_index(drop=True)

bec_results = []
for i in tqdm(range(0, len(bec), batch_size), desc="Scoring BEC"):
    batch = bec['text'].iloc[i:i+batch_size].tolist()
    inputs = tokenizer_inf(batch, truncation=True,
                           padding='max_length', max_length=256,
                           return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model_inf(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    for prob in probs:
        bec_results.append({
            'llm_label': int(np.argmax(prob)),
            'conf_legitimate': float(prob[0]),
            'conf_human_phishing': float(prob[1]),
            'conf_ai_phishing': float(prob[2])
        })

bec_df = pd.DataFrame(bec_results)
bec_df.to_csv(DATA_PROCESSED / "bec_modernbert_cleaned.csv", index=False)

flagged = (bec_df['llm_label'] != 0).sum()
print(f"\nBEC External Test (Cleaned Model):")
print(bec_df['llm_label'].value_counts().sort_index())
print(f"\nFlagged as phishing: {flagged}/{len(bec_df)} ({flagged/len(bec_df)*100:.1f}%)")
print(f"Previously (original model): 6.1%")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip uninstall torchvision torchaudio -y -q
!pip install transformers datasets accelerate -q

import pandas as pd
import numpy as np
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import Dataset

BASE_DIR = Path("/content/drive/MyDrive/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")

# Load diverse dataset
dataset = pd.read_csv(DATA_PROCESSED / "dataset_diverse.csv")
print(f"Diverse dataset: {len(dataset)} emails")
print(dataset['label'].value_counts().sort_index())

# Split
train_df, temp_df = train_test_split(
    dataset, test_size=0.30, random_state=42, stratify=dataset['label'])
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])
print(f"\nTrain: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Tokenize
MODEL_NAME = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    tokens = tokenizer(examples['text'], truncation=True,
                       padding='max_length', max_length=256)
    tokens['labels'] = examples['label']
    return tokens

train_hf = Dataset.from_dict({'text': train_df['text'].tolist(),
                               'label': train_df['label'].astype(int).tolist()})
val_hf = Dataset.from_dict({'text': val_df['text'].tolist(),
                             'label': val_df['label'].astype(int).tolist()})

train_hf = train_hf.map(preprocess, batched=True, remove_columns=['text','label'])
val_hf = val_hf.map(preprocess, batched=True, remove_columns=['text','label'])
train_hf.set_format('torch')
val_hf.set_format('torch')

# Forward pass test
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True).to('cuda')

sample = train_hf[0]
with torch.no_grad():
    out = model(input_ids=sample['input_ids'].unsqueeze(0).to('cuda'),
                attention_mask=sample['attention_mask'].unsqueeze(0).to('cuda'),
                labels=sample['labels'].unsqueeze(0).to('cuda'))
print(f"\nForward pass loss: {out.loss:.4f}")
print("Ready to train!" if not torch.isnan(out.loss) else "WARNING: NaN")

# Train
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'macro_f1': f1_score(labels, preds, average='macro'),
            'accuracy': (preds == labels).mean()}

training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / "modernbert_diverse_checkpoints"),
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=100,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_hf, eval_dataset=val_hf,
    compute_metrics=compute_metrics)

print("\nTraining ModernBERT on diverse dataset")
trainer.train()

print("\nTraining complete!")
metrics = trainer.evaluate()
print("\nFinal validation metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

# Save
save_path = str(MODELS_DIR / "modernbert_diverse")
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

# Verify
saved = Path(save_path)
print(f"\nSave verification:")
print(f"  Folder exists: {saved.exists()}")
for f in saved.iterdir():
    print(f"  {f.name}")

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

BASE_DIR = Path("/content/drive/MyDrive/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"

# Load diverse model
model_path = str(MODELS_DIR / "modernbert_diverse")
model_inf = AutoModelForSequenceClassification.from_pretrained(
    model_path, local_files_only=True).to('cuda')
tokenizer_inf = AutoTokenizer.from_pretrained(
    model_path, local_files_only=True)
model_inf.eval()
print("Diverse ModernBERT loaded")

# Score full diverse dataset
dataset = pd.read_csv(DATA_PROCESSED / "dataset_diverse.csv")
print(f"Scoring {len(dataset)} emails")

results = []
batch_size = 32

for i in tqdm(range(0, len(dataset), batch_size), desc="Scoring dataset"):
    batch_texts = dataset['text'].iloc[i:i+batch_size].tolist()
    batch_labels = dataset['label'].iloc[i:i+batch_size].tolist()
    inputs = tokenizer_inf(batch_texts, truncation=True,
                           padding='max_length', max_length=256,
                           return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model_inf(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    for j, (prob, label) in enumerate(zip(probs, batch_labels)):
        results.append({
            'email_idx': i+j, 'true_label': label,
            'llm_label': int(np.argmax(prob)),
            'conf_legitimate': float(prob[0]),
            'conf_human_phishing': float(prob[1]),
            'conf_ai_phishing': float(prob[2])
        })

results_df = pd.DataFrame(results)
results_df.to_csv(DATA_PROCESSED / "modernbert_diverse_features.csv", index=False)
print(f"Dataset scored: {len(results_df)} emails")

# Score held-out BEC test set
bec_test = pd.read_csv(DATA_PROCESSED / "bec_test_holdout.csv")
bec_test['text'] = bec_test['text'].astype(str)
print(f"\nScoring BEC holdout: {len(bec_test)} emails")

bec_results = []
for i in tqdm(range(0, len(bec_test), batch_size), desc="Scoring BEC holdout"):
    batch = bec_test['text'].iloc[i:i+batch_size].tolist()
    inputs = tokenizer_inf(batch, truncation=True,
                           padding='max_length', max_length=256,
                           return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model_inf(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    for prob in probs:
        bec_results.append({
            'llm_label': int(np.argmax(prob)),
            'conf_legitimate': float(prob[0]),
            'conf_human_phishing': float(prob[1]),
            'conf_ai_phishing': float(prob[2])
        })

bec_df = pd.DataFrame(bec_results)
bec_df.to_csv(DATA_PROCESSED / "bec_diverse_modernbert.csv", index=False)

flagged = (bec_df['llm_label'] != 0).sum()
print(f"\nBEC Holdout Test (Diverse Model):")
print(bec_df['llm_label'].value_counts().sort_index())
print(f"\nFlagged as phishing: {flagged}/{len(bec_df)} ({flagged/len(bec_df)*100:.1f}%)")
print(f"\nComparison so far:")
print(f"  Original model on full BEC:   6.1%")
print(f"  Cleaned model on full BEC:    10.8%")
print(f"  Diverse model on BEC holdout: {flagged/len(bec_df)*100:.1f}%")

In [ ]:
# Score held-out BEC test set
bec_test = pd.read_csv(DATA_PROCESSED / "bec_test_holdout.csv")
bec_test['text'] = bec_test['text'].astype(str)
print(f"Scoring BEC holdout: {len(bec_test)} emails...")

bec_results = []
batch_size = 32

for i in tqdm(range(0, len(bec_test), batch_size), desc="Scoring BEC holdout"):
    batch = bec_test['text'].iloc[i:i+batch_size].tolist()
    inputs = tokenizer_inf(batch, truncation=True,
                           padding='max_length', max_length=256,
                           return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model_inf(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    for prob in probs:
        bec_results.append({
            'llm_label': int(np.argmax(prob)),
            'conf_legitimate': float(prob[0]),
            'conf_human_phishing': float(prob[1]),
            'conf_ai_phishing': float(prob[2])
        })

bec_df = pd.DataFrame(bec_results)
bec_df.to_csv(DATA_PROCESSED / "bec_diverse_modernbert.csv", index=False)

flagged = (bec_df['llm_label'] != 0).sum()
print(f"\nBEC Holdout Test (Diverse Model):")
print(bec_df['llm_label'].value_counts().sort_index())
print(f"\nFlagged as phishing: {flagged}/{len(bec_df)} ({flagged/len(bec_df)*100:.1f}%)")
print(f"\nComparison:")
print(f"  Original model on full BEC:   6.1%")
print(f"  Cleaned model on full BEC:    10.8%")
print(f"  Diverse model on BEC holdout: {flagged/len(bec_df)*100:.1f}%")